# 03 — TF-IDF
**Goal:** Ponderar cada término por su importancia real — frecuente en el documento pero raro en el corpus.

## La matemática

$$\text{TF-IDF}(t, d) = \underbrace{\frac{f_{t,d}}{\sum_k f_{k,d}}}_{\text{TF}} \times \underbrace{\log\frac{N+1}{df_t+1} + 1}_{\text{IDF}}$$

- **TF** (Term Frequency): qué tan frecuente es el término en *este* documento
- **IDF** (Inverse Document Frequency): qué tan raro es en el *corpus completo*
- Un término que aparece en todos los docs tiene IDF ≈ 0 → peso bajo

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
})

STOPWORDS_ES = [
    'a','al','algo','ante','como','con','de','del','desde','el','ella','en',
    'entre','es','esta','este','esto','fue','la','las','le','lo','los','me',
    'mi','muy','no','nos','o','para','pero','por','que','se','si','sin',
    'su','sus','también','te','todo','un','una','y','ya','yo',
]

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\wáéíóúüñ\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

In [ ]:
templates_pos = [
    'El proceso de solicitud fue muy rápido y eficiente',
    'Excelente servicio, aprobaron mi tarjeta rápidamente',
    'La app funciona perfecto, muy fácil de usar',
    'Increíbles beneficios, recomiendo la tarjeta al 100',
    'Atención al cliente excepcional, resolvieron todo rápido',
    'Proceso de activación muy sencillo y sin problemas',
]
templates_neg = [
    'La app está muy lenta, tardé mucho en activar la tarjeta',
    'No me llegó el OTP, tuve que llamar varias veces',
    'La carga de documentos falla constantemente',
    'Llevo días esperando la evaluación crediticia sin respuesta',
    'El call center no responde, pésimo servicio al cliente',
    'El proceso de aprobación es muy lento e ineficiente',
]

rng = np.random.default_rng(42)
comments  = [templates_pos[rng.integers(6)] for _ in range(300)] + \
            [templates_neg[rng.integers(6)] for _ in range(300)]
sentiments = [1]*300 + [0]*300

df = pd.DataFrame({'comment': comments, 'sentiment': sentiments})
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## 1. TF-IDF a mano — para internalizar la fórmula

In [ ]:
docs = [
    'la app es lenta y falla',
    'el proceso fue rápido y eficiente',
    'la app es rápida y fácil',
    'el proceso de evaluación falla siempre',
]

# TF: conteo normalizado por longitud del doc
def tf(doc):
    tokens = doc.split()
    counts = {}
    for t in tokens:
        counts[t] = counts.get(t, 0) + 1
    return {t: c/len(tokens) for t, c in counts.items()}

# IDF: log((N+1)/(df+1)) + 1  (fórmula de sklearn smooth IDF)
def idf(docs):
    N = len(docs)
    df_count = {}
    for doc in docs:
        for t in set(doc.split()):
            df_count[t] = df_count.get(t, 0) + 1
    return {t: np.log((N+1)/(c+1))+1 for t, c in df_count.items()}

idf_vals = idf(docs)
print('IDF por término (mayor = más raro):')
for word, val in sorted(idf_vals.items(), key=lambda x: -x[1]):
    print(f'  {word:15s} {val:.3f}')

In [ ]:
# TF-IDF del primer doc
doc0 = docs[0]
tf0  = tf(doc0)
print(f'Doc: "{doc0}"\n')
print(f'{"token":15s} {"TF":>8s} {"IDF":>8s} {"TF-IDF":>8s}')
print('-' * 45)
for token in sorted(tf0):
    t   = tf0[token]
    i   = idf_vals.get(token, 0)
    print(f'{token:15s} {t:8.3f} {i:8.3f} {t*i:8.3f}')

## 2. TfidfVectorizer de sklearn

In [ ]:
tfidf = TfidfVectorizer(
    preprocessor=preprocess,
    stop_words=STOPWORDS_ES,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2),
    sublinear_tf=True,    # TF → log(1+TF) — reduce efecto de repetición extrema
)

X = tfidf.fit_transform(df['comment'])   # sparse (600, vocab)
feature_names = tfidf.get_feature_names_out()

print(f'Shape: {X.shape}')
print(f'Vocabulario: {len(feature_names)} tokens')

In [ ]:
# IDF de cada término (disponible después de fit)
idf_learned = tfidf.idf_

# Los términos con IDF más alto son los más discriminativos
top_idf_idx = np.argsort(idf_learned)[::-1][:15]
low_idf_idx = np.argsort(idf_learned)[:15]

print('Tokens con IDF más ALTO (raros / específicos):')
for i in top_idf_idx:
    print(f'  {feature_names[i]:30s} IDF={idf_learned[i]:.3f}')

print('\nTokens con IDF más BAJO (frecuentes / genéricos):')
for i in low_idf_idx:
    print(f'  {feature_names[i]:30s} IDF={idf_learned[i]:.3f}')

## 3. TF-IDF vs BoW — comparación práctica

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

y = df['sentiment'].to_numpy()

pipe_bow = Pipeline([
    ('vec', CountVectorizer(preprocessor=preprocess, stop_words=STOPWORDS_ES,
                            min_df=2, ngram_range=(1,2))),
    ('clf', LogisticRegression(max_iter=1000)),
])

pipe_tfidf = Pipeline([
    ('vec', TfidfVectorizer(preprocessor=preprocess, stop_words=STOPWORDS_ES,
                            min_df=2, ngram_range=(1,2), sublinear_tf=True)),
    ('clf', LogisticRegression(max_iter=1000)),
])

for name, pipe in [('BoW + LR', pipe_bow), ('TF-IDF + LR', pipe_tfidf)]:
    scores = cross_val_score(pipe, df['comment'], y, cv=5, scoring='f1')
    print(f'{name:20s}  F1 = {scores.mean():.4f} ± {scores.std():.4f}')

## 4. Features más importantes por clase

In [ ]:
# Entrenar LR y examinar coeficientes
pipe_tfidf.fit(df['comment'], y)
coefs = pipe_tfidf.named_steps['clf'].coef_[0]
fn    = pipe_tfidf.named_steps['vec'].get_feature_names_out()

top_pos = np.argsort(coefs)[::-1][:12]
top_neg = np.argsort(coefs)[:12]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].barh(fn[top_pos][::-1], coefs[top_pos][::-1], color='#06d6a0')
axes[0].set_title('Features → POSITIVO')
axes[0].set_xlabel('Coeficiente LR')

axes[1].barh(fn[top_neg], np.abs(coefs[top_neg]), color='#f72585')
axes[1].set_title('Features → NEGATIVO')
axes[1].set_xlabel('|Coeficiente LR|')

plt.suptitle('TF-IDF + Logistic Regression — top features por clase', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Parámetros clave de TfidfVectorizer

| Parámetro | Default | Cuándo cambiar |
|---|---|---|
| `ngram_range=(1,1)` | unigramas | `(1,2)` para capturar frases clave |
| `min_df=1` | todos los tokens | subir a `2`–`5` en corpus grande |
| `max_df=1.0` | sin límite | bajar a `0.9` para eliminar términos ubicuos |
| `sublinear_tf=False` | TF lineal | `True` para documentos largos |
| `max_features=None` | vocab completo | limitar a `10_000`–`50_000` para RAM |

## Resumen
| Concepto | Fórmula / API |
|---|---|
| TF | frecuencia del término en el doc (normalizada) |
| IDF | `log((N+1)/(df+1)) + 1` — penaliza términos ubicuos |
| TF-IDF | `TF × IDF` — alto si frecuente aquí y raro globalmente |
| `TfidfVectorizer` | `fit_transform(corpus)` → sparse matrix |
| `tfidf.idf_` | IDF aprendido para cada token |
| `sublinear_tf=True` | `log(1+tf)` — recomendado en docs largos |

**Siguiente:** `04_similarity.ipynb` — medir distancia entre documentos con similitud coseno.